### Dependencies

In [ ]:
import os
import re
import sys
import time
import emoji
import fasttext
import numpy as np
import pandas as pd
import seaborn as sns
import urllib.request
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.axes import Axes
from wordcloud import WordCloud
from scipy.stats import pearsonr
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from deep_translator import GoogleTranslator
from langdetect import detect, LangDetectException
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer


In [ ]:
# Ensure project root is on sys.path so mfs_sentiment is importable from notebooks/
_project_root = Path.cwd().resolve()
for parent in [_project_root, *_project_root.parents]:
    if (parent / "requirements.txt").exists():
        _project_root = parent
        break
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

### Configs

In [ ]:
from mfs_sentiment.config import (
    PROJECT_ROOT,
    DATASET_DIR,
    FASTTEXT_MODEL_DIR,
    FASTTEXT_MODEL_PATH,
    FASTTEXT_MODEL_NAME,
    RESULT_DIR,
    TRANSLATED_CSV_DIR,
    CLEANED_CSV_DIR,
    DATASET_PATH_DICT,
    MASTER_CSV_PATH,
    SCORED_MASTER_CSV_PATH,
    REQUIRED_RAW_CSV_COLUMNS,
    REQUIRED_MASTER_CSV_COLUMNS,
    REQUIRED_SCORED_MASTER_CSV_COLUMNS,
    STOPWORDS_FOR_LDA,
    LDA_MIN_THRESHOLD,
    LDA_TOPIC_NUM,
    LDA_TOPWORD_NUM,
    ensure_dirs,
)

from mfs_sentiment.models import ensure_fasttext_model, get_fasttext_model
from mfs_sentiment.dictionaries import PRE_TRANSLATION_DICT, POST_TRANSLATION_DICT, APP_SPECIFIC_TRANSLATION_DICT
ensure_dirs()
ensure_fasttext_model()

### Standardizing the Reviews

In [ ]:
# def is_banglish(text: str) -> bool:
#     """Return True when a review contains known Bangla-English mixed keywords."""
#     if not isinstance(text, str):
#         return False
#     words = set(text.lower().split())
#     return bool(words & BANGLISH_KEYWORDS)

In [ ]:
def apply_custom_dict(text: str, translation_dict: dict) -> str:
    """Replace known tokens in text using a custom translation lookup."""
    if not isinstance(text, str):
        return ""
    for bangla, english in translation_dict.items():
        text = text.replace(bangla, english)
    return text

In [ ]:
def convert_emojis(text: str) -> str:
    """Replace emojis with their textual descriptions for downstream normalization."""
    if not isinstance(text, str):
        return ""
    return emoji.demojize(text, delimiters=(" ", " "))

In [ ]:
def detect_language(text: str) -> tuple:
    """Detect the language of a review using FastText with a langdetect fallback."""
    _FT_MODEL = get_fasttext_model()
    try:
        label, confidence = _FT_MODEL.predict(text.replace("\n", " "), k=1)
        lang = label[0].replace("__label__", "")  # type: ignore
        return lang, float(confidence[0])
    except Exception:
        pass

    try:
        return detect(text), 0.0
    except LangDetectException:
        return "unknown", 0.0

In [ ]:
def clean_text(text: str) -> str:
    """Normalize review text by lowercasing, removing URLs, and collapsing whitespace."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"https?://\S+|www\.\S+", "", text)  # Remove URLs
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)  # Collapse repeated characters
    text = re.sub(r"[^a-z0-9\s:_.,!?&'()-]", "", text)  # Keep letters, digits, emoji tags
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
def translate_to_english(text: str, source_lang: str, confidence: float, only_english: bool = False) -> str:
    """Translate non-English input to English, unless translation isn't needed.

    Parameters
    ----------
    text : str
        Text to translate.
    source_lang : str
        Language code detected for `text` (e.g. 'en').
    confidence : float
        Confidence score for the language detection, in [0, 1].
    only_english : bool, optional
        If True, skip translation entirely: rows that aren't English will be
        dropped downstream by `build_standarized_csv`, and English rows don't
        need translating. Default is False (translate as usual).

    Returns
    -------
    str
        The English text, or the original text if translation was skipped
        (either because it's already confidently English, or only_english=True).
    """
    if not text.strip():
        return text

    if only_english:
        # Non-English rows get filtered out later; English rows need no translation.
        return text

    is_confident_english = (
        source_lang == "en"
        and text.isascii()
        and confidence > 0.90
        # and not is_banglish(text)
    )
    if is_confident_english:
        return text

    try:
        result = GoogleTranslator(source="auto", target="en").translate(text)
        time.sleep(0.3)
        return result
    except Exception as e:
        print(f"Translation error: {e}")
        return text

In [ ]:
def _run_standardization_steps(text: str, app_name: str, only_english: bool = False) -> dict:
    """Normalize a single review through the full standardization pipeline.

    Applies app-specific translations, emoji normalization, language detection,
    translation to English (unless only_english=True), cleaning, and a final
    post-processing pass, returning the standardized text plus intermediate
    artifacts used for inspection.

    Parameters
    ----------
    text : str
        Raw review text.
    app_name : str
        App the review belongs to; selects the app-specific translation dict.
    only_english : bool, optional
        If True, skip the actual translation call (see translate_to_english).
        Default is False.

    Returns
    -------
    dict
        Keys: original_text, pre_translated_text, emoji_converted, detected_lang,
        confidence, translated_text, cleaned_text, post_translated (final text).
    """
    text = text.lower()
    pre_translated = apply_custom_dict(text, PRE_TRANSLATION_DICT | APP_SPECIFIC_TRANSLATION_DICT[app_name])
    emoji_converted = convert_emojis(pre_translated)
    detected_lang, confidence = detect_language(emoji_converted)
    translated = translate_to_english(emoji_converted, detected_lang, confidence, only_english=only_english)
    cleaned = clean_text(translated)
    post_translated = apply_custom_dict(cleaned, POST_TRANSLATION_DICT)

    return {
        "original_text": text,
        "pre_translated_text": pre_translated,
        "emoji_converted": emoji_converted,
        "detected_lang": detected_lang,
        "confidence": round(float(confidence), 3),
        "translated_text": translated,
        "cleaned_text": cleaned,
        "post_translated": post_translated,
    }

### Data Loading & Cleaning

In [ ]:
def build_standarized_csv(input_path: str|Path, only_english: bool = False) -> Path:
    """Clean, standardize, and save review data from a raw CSV file.

    Reads the raw reviews, keeps required columns, fills missing app versions,
    runs the normalization pipeline row-by-row, and writes both the translated
    intermediate results and the cleaned dataset to CLEANED_CSV_DIR (a temp dir).

    Parameters
    ----------
    input_path : str
        Path to a raw per-app review csv (see DATASET_PATH_DICT).
    only_english : bool, optional
        If True, translation is skipped and only reviews detected as English
        are kept in the cleaned output. If False (default), all reviews are
        kept, with non-English ones translated to English.

    Returns
    -------
    str
        Path to the cleaned csv, written under CLEANED_CSV_DIR.
    """
    df = pd.read_csv(input_path)
    df = df[REQUIRED_RAW_CSV_COLUMNS].copy()
    df = df.dropna(subset=['review_text'])
    df['app_version'] = df['app_version'].fillna('unknown')

    step_results = df.apply(
        lambda row: _run_standardization_steps(row['review_text'], row['app_name'], only_english=only_english),
        axis=1,
    )
    steps_df = pd.DataFrame(step_results.tolist())
    steps_df.insert(0, 'review_id', df['review_id'].values.tolist())

    if only_english:
        # Rows that weren't translated (non-English) are dropped here.
        english_mask = (steps_df['detected_lang'] == 'en').values
        df = df[english_mask].reset_index(drop=True)
        steps_df = steps_df[english_mask].reset_index(drop=True)

    base, ext = os.path.splitext(os.path.basename(input_path))
    steps_df.to_csv(TRANSLATED_CSV_DIR / f"{base}_translated{ext}", index=False)

    idx = df.columns.get_loc('review_text')
    df.insert(idx + 1, 'review_text_clean', steps_df['post_translated'].values)  # type: ignore

    output_path = CLEANED_CSV_DIR / f"{base}_cleaned{ext}"
    df.to_csv(output_path, index=False)
    return output_path

### Finding Sentiment Score

In [ ]:
def generate_sentiment_scores(input_path: str|Path, output_path: str|Path|None = None) -> str|Path:
    """
    Calculate sentiment scores for review texts using the VADER sentiment analyzer.

    Parameters
    ----------
    input_path : str
        Path to input CSV file (must contain 'review_text_clean' column).
    output_path : str, optional
        Where to write the scored csv. If None, saves alongside input_path
        with a '_scored' suffix. Default is None.

    Returns
    -------
    str
        Path to the output CSV file with sentiment scores.
    """
    df = pd.read_csv(input_path)

    analyzer = SentimentIntensityAnalyzer()
    df["sentiment_score"] = df["review_text_clean"].apply(
        lambda text: analyzer.polarity_scores(str(text))["compound"]
    )

    if output_path is None:
        base, ext = os.path.splitext(input_path)
        output_path = f"{base}_scored{ext}"
    df.to_csv(output_path, index=False)
    return output_path

### Calculating Gap Scores

In [ ]:
def _compute_gap_score(sentiment_score: float, rating: int|float) -> float:
    """
    Calculate the gap score between normalized star rating and sentiment score.
    
    Measures the mismatch between user's star rating and the actual sentiment
    expressed in their review text. Normalizes both to [-1, 1] range and computes
    the difference normalized to [-1, 1].
    
    Parameters
    ----------
    sentiment_score : float
        VADER compound sentiment score in range [-1.0, 1.0].
    rating : int or float
        User's star rating in range [1, 5].
    
    Returns
    -------
    float
        Gap score in range [-1.0, 1.0].
        Positive = rating higher than sentiment (inflated rating)
        Negative = rating lower than sentiment (deflated rating)
        Near 0 = rating consistent with sentiment
    
    Notes
    -----
    - Ratings [1, 5] are mapped to [-1, 1] range via (rating - 3.0) / 2.0
    - Gap = (normalized_rating - sentiment_score) / 2.0
    """
    normalized_star = (rating - 3.0) / 2.0                  # maps [1, 5] to [-1, 1]
    gap_score = (normalized_star - sentiment_score) / 2.0   # Calculating gap score [-1, 1]   
    return gap_score

In [ ]:
def _classify_gap_score(gap_score: float) -> str:
    """
    Classify gap score into a mismatch category.
    
    Categorizes the gap score into one of three labels based on thresholds:
    - Inflated: user gave a higher rating than sentiment suggests
    - Deflated: user gave a lower rating than sentiment suggests
    - Consistent: rating and sentiment are aligned
    
    Parameters
    ----------
    gap_score : float
        Gap score in range [-1.0, 1.0] (output from _compute_gap_score).
    
    Returns
    -------
    str
        One of: 'Inflated', 'Deflated', or 'Consistent'.
        - 'Inflated': gap_score > 0.25
        - 'Deflated': gap_score < -0.25
        - 'Consistent': gap_score in [-0.25, 0.25]
    
    Notes
    -----
    - Thresholds (±0.25) provide a tolerance for small mismatches
    """
    if gap_score > 0.25:
        return "Inflated"
    elif gap_score < -0.25:
        return "Deflated"
    else:
        return "Consistent"

In [ ]:
def generate_gap_scores(input_path: str|Path) -> pd.DataFrame:
    """
    Compute gap scores and classify rating-sentiment mismatches for all reviews.
    
    Calculates gap score for each review by comparing normalized star rating with
    sentiment score, then classifies each review as Inflated/Deflated/Consistent.
    Adds 'gap_score' and 'mismatch_label' columns and saves updated data back to input file.
    
    Parameters
    ----------
    input_path : str
        Path to CSV file with reviews (must contain 'rating' and 'sentiment_score' columns).
    
    Returns
    -------
    pd.DataFrame
        DataFrame with all original columns plus 'gap_score' and 'mismatch_label'.
        Data is also saved back to the input CSV file.
    
    Notes
    -----
    - Converts 'rating' to int and 'sentiment_score' to float before computation
    - Uses _compute_gap_score() to calculate gap for each row
    - Uses _classify_gap_score() to assign mismatch labels
    - Modifies and saves the input CSV file in-place
    """
    df = pd.read_csv(input_path)

    # ensuring correct dtypes before computation
    df["rating"] = df["rating"].astype(int)
    df["sentiment_score"] = df["sentiment_score"].astype(float)

    # Computing gap score
    df["gap_score"] = df.apply(
        lambda row: _compute_gap_score(row["sentiment_score"], row["rating"]),
        axis=1
    )

    # Assigning mismatch label (ratings w.r.t sentiment) > inflated/deflated/consistent
    df["mismatch_label"] = df["gap_score"].apply(_classify_gap_score)

    df.to_csv(input_path, index=False)
    return df

### Creating a master csv & Scoring it

In [ ]:
def build_master_csv(only_english: bool = False) -> pd.DataFrame:
    """Build the combined, unscored master CSV from all apps' cleaned reviews.

    Cleans each app's raw csv (see build_standarized_csv), concatenates the
    results, and saves the combined dataset to MASTER_CSV_PATH. Sentiment and
    gap scores are NOT computed here — run score_master_csv() afterward.

    Parameters
    ----------
    only_english : bool, optional
        Passed through to build_standarized_csv for each app. Default is False.

    Returns
    -------
    pd.DataFrame
        The combined, cleaned (unscored) master dataframe.
    """
    cleaned_frames = []
    for _, raw_path in DATASET_PATH_DICT.items():
        cleaned_path = build_standarized_csv(raw_path, only_english=only_english)
        cleaned_frames.append(pd.read_csv(cleaned_path))

    master_df = pd.concat(cleaned_frames, ignore_index=True)
    master_df.to_csv(MASTER_CSV_PATH, index=False)
    return master_df

In [ ]:
def load_master_csv(path: Path|str, required_columns: list = REQUIRED_SCORED_MASTER_CSV_COLUMNS) -> pd.DataFrame | None:
    """
    Load and validate a master CSV file against a set of required columns.

    Parameters
    ----------
    path : str
        Path to the master CSV file to load.
    required_columns : list of str, optional
        Columns that must be present. Default is REQUIRED_SCORED_MASTER_CSV_COLUMNS
        (use REQUIRED_MASTER_CSV_COLUMNS for the pre-scoring master csv).

    Returns
    -------
    pd.DataFrame or None
        The loaded dataframe, or None if any required column is missing.
    """
    df = pd.read_csv(path)
    has_required_columns = set(required_columns).issubset(df.columns)
    if not has_required_columns:
        print("Error: The csv file must have the following columns: ", required_columns)
        return None
    return df

In [ ]:
def score_master_csv(master_csv_path: Path|str = MASTER_CSV_PATH) -> pd.DataFrame:
    """Compute sentiment and gap scores for the master CSV and save the result.

    Loads the unscored master csv, runs VADER sentiment scoring on
    'review_text_clean', computes gap scores and mismatch labels, and writes
    the fully scored dataset to SCORED_MASTER_CSV_PATH.

    Parameters
    ----------
    master_csv_path : str, optional
        Path to the unscored master csv. Default is MASTER_CSV_PATH.

    Returns
    -------
    pd.DataFrame
        The scored master dataframe (also saved to SCORED_MASTER_CSV_PATH).
    """
    scored_path = generate_sentiment_scores(master_csv_path, output_path=SCORED_MASTER_CSV_PATH)
    return generate_gap_scores(scored_path)

### Analysis

#### App-level Statistics

In [ ]:
def app_level_summary(df: pd.DataFrame) -> pd.DataFrame:
    """
    Generate aggregated statistics for each app across all reviews.
    
    Computes per-app metrics including review count, gap score statistics,
    mismatch category distributions, and correlation between normalized ratings
    and sentiment scores.
    
    Parameters
    ----------
    df : pd.DataFrame
        Master dataframe with all reviews (must contain columns: 'app_name',
        'rating', 'gap_score', 'mismatch_label', 'sentiment_score').
    
    Returns
    -------
    pd.DataFrame
        Summary statistics per app with columns:
        - app_name: app identifier
        - n_reviews: number of reviews for that app
        - mean_gap_score: average gap score
        - pct_inflated: percentage of reviews with 'Inflated' label
        - pct_deflated: percentage of reviews with 'Deflated' label
        - pct_consistent: percentage of reviews with 'Consistent' label
        - pearson_r: Pearson correlation coefficient
        - pearson_p_value: p-value for the correlation test
    
    Notes
    -----
    - Ratings are normalized to [-1, 1] range for correlation calculation
    - Rows with NaN sentiment_score or rating are excluded from correlation
    - Correlations test relationship between normalized rating and actual sentiment
    """
    summary_rows = []

    for app, group in df.groupby("app_name"):
        normalized_rating = (group["rating"] - 3) / 2

        valid = normalized_rating.notna() & group["sentiment_score"].notna()
        valid_ratings = normalized_rating[valid]
        valid_sentiment = group["sentiment_score"][valid]
        if valid.sum() < 2 or valid_ratings.nunique() < 2 or valid_sentiment.nunique() < 2:
            r_value, p_value = float("nan"), float("nan")
        else:
            r_value, p_value = pearsonr(valid_ratings, valid_sentiment)

        label_pct = group["mismatch_label"].value_counts(normalize=True) * 100

        summary_rows.append({
            "app_name": app,
            "n_reviews": len(group),
            "mean_gap_score": group["gap_score"].mean(),
            "pct_inflated": label_pct.get("Inflated", 0.0),
            "pct_deflated": label_pct.get("Deflated", 0.0),
            "pct_consistent": label_pct.get("Consistent", 0.0),
            "pearson_r": r_value,
            "pearson_p_value": p_value,
        })

    return pd.DataFrame(summary_rows)

In [ ]:
def monthly_gap_trend(df: pd.DataFrame, warning: bool = False) -> pd.DataFrame:
    """
    Calculate monthly average gap scores per app for temporal trend analysis.
    
    Aggregates gap scores by app and calendar month to show how rating-sentiment
    mismatches evolve over time. Useful for time-series visualization (heatmaps,
    line charts).
    
    Parameters
    ----------
    df : pd.DataFrame
        Master dataframe with all reviews (must contain 'app_name',
        'review_date', and 'gap_score' columns).
    warning : bool, optional
        If True, print warning messages about unparseable dates. Default is False.
    
    Returns
    -------
    pd.DataFrame
        Monthly aggregated data with columns:
        - app_name: app identifier
        - year_month: calendar month as Period object (YYYY-MM format)
        - mean_gap_score: average gap score for that app in that month
    
    Notes
    -----
    - review_date is parsed with errors='coerce' (unparseable dates become NaT)
    - Unparseable rows are dropped before aggregation
    - Useful for detecting seasonal patterns or event-driven trends in mismatches
    - If warning=True, reports count of unparseable dates
    """
    df = df.copy()
    df["review_date"] = pd.to_datetime(df["review_date"], errors="coerce")

    if warning:
        n_bad_dates = df["review_date"].isna().sum()
        if n_bad_dates:
            print(f"[WARNING] {n_bad_dates} rows have unparseable review_date")

    temp = df.dropna(subset=["review_date"]).copy()
    temp["year_month"] = temp["review_date"].dt.to_period("M")

    trend = (
        temp.groupby(["app_name", "year_month"])["gap_score"]
        .mean()
        .reset_index()
        .rename(columns={"gap_score": "mean_gap_score"})
    )
    return trend

#### Topic Modeling

In [ ]:
# Text preprocessing for topic modeling > LDA needs tokenized, stopword-filtered, lemmatized text
_lemmatizer = WordNetLemmatizer() 
_custom_stopwords = set(stopwords.words("english")) | STOPWORDS_FOR_LDA

def _preprocess_for_lda(text: str) -> str:
    """
    Preprocess review text for Latent Dirichlet Allocation (LDA) topic modeling.
    
    Performs multi-step text cleaning: tokenization, alphabetic filtering,
    stopword removal, and lemmatization to prepare text for LDA analysis.
    
    Parameters
    ----------
    text : str
        Raw review text to preprocess.
    
    Returns
    -------
    str
        Preprocessed text as space-separated tokens ready for LDA.
        Tokens are lowercase, lemmatized, and free of stopwords and punctuation.
    
    Notes
    -----
    - Converts text to lowercase during tokenization
    - Removes non-alphabetic characters (punctuation, numbers, symbols)
    - Filters out English stopwords plus custom domain stopwords (STOPWORDS_FOR_LDA)
    - Lemmatizes tokens to root form (e.g., 'running' → 'run')
    - Requires NLTK tokenizer, lemmatizer, and stopwords to be downloaded
    - Use sparingly; preprocessing can be time-intensive for large datasets
    """
    tokens = word_tokenize(str(text).lower())
    tokens = [t for t in tokens if t.isalpha()]                 # Discarding punctuation/numbers
    tokens = [t for t in tokens if t not in _custom_stopwords]  # Removing stopwords
    tokens = [_lemmatizer.lemmatize(t) for t in tokens]         # Converting words to root form: running -> run
    return " ".join(tokens)

In [ ]:
def run_lda_on_subset(df: pd.DataFrame, label: str, n_topics: int = 5, n_top_words: int = 10):
    """
    Perform Latent Dirichlet Allocation (LDA) topic modeling on a mismatch subset.
    
    Filters reviews by mismatch label (Inflated/Deflated/Consistent), preprocesses
    text, builds a CountVectorizer and LDA model, and extracts top words per topic.
    Each subset gets its own fitted vectorizer and LDA model (not shared).
    
    Parameters
    ----------
    df : pd.DataFrame
        Master dataframe with all reviews (must contain 'review_text_clean'
        and 'mismatch_label' columns).
    label : str
        Mismatch label to filter on: 'Inflated', 'Deflated', or 'Consistent'.
    n_topics : int, optional
        Number of topics for LDA model. Default is 5.
    n_top_words : int, optional
        Number of top words to extract per topic. Default is 10.
    
    Returns
    -------
    tuple(lda_model, topics) or (None, None)
        lda_model : sklearn.decomposition.LatentDirichletAllocation or None
            Fitted LDA model. None if subset is empty.
        topics : list of dict or None
            List of dictionaries with format:
            [{'topic': int, 'top_words': [str, ...]}, ...]
            None if subset is empty.
    
    Notes
    -----
    - Preprocesses text using preprocess_for_lda() before vectorization
    - CountVectorizer: max_df=0.9 (ignore tokens in >90% docs),
      min_df=LDA_MIN_THRESHOLD (ignore tokens in fewer than that many docs)
    - LDA uses random_state=42 for reproducibility
    - Separate vectorizer per subset ensures vocabulary reflects mismatch-specific themes
    - Prints warning if label not found in DataFrame
    """
    subset = df[df["mismatch_label"] == label].copy()

    if subset.empty:
        print(f"[WARNING] No rows found for label '{label}'")
        return None, None

    if len(subset) < LDA_MIN_THRESHOLD:
        print(
            f"[WARNING] Too few rows for LDA on label '{label}' "
            f"({len(subset)} < {LDA_MIN_THRESHOLD})"
        )
        return None, None

    subset["lda_ready_text"] = subset["review_text_clean"].apply(_preprocess_for_lda)

    vectorizer = CountVectorizer(max_df=0.9, min_df=LDA_MIN_THRESHOLD)
    doc_term_matrix = vectorizer.fit_transform(subset["lda_ready_text"])

    lda_model = LatentDirichletAllocation(
        n_components=n_topics, random_state=42
    )
    lda_model.fit(doc_term_matrix)

    feature_names = vectorizer.get_feature_names_out()
    topics = []
    for topic_idx, topic in enumerate(lda_model.components_):
        top_indices = topic.argsort()[-n_top_words:][::-1]
        top_words = [feature_names[i] for i in top_indices]
        top_weights = [float(topic[i]) for i in top_indices]
        topics.append({"topic": topic_idx, "top_words": top_words, "top_weights": top_weights,})

    return lda_model, topics

#### Running full analysis

In [ ]:
def run_full_analysis(master_csv_path: Path|str, verbose: bool = True, lda_per_app: bool = True):
    """
    Execute complete analysis pipeline including app-level stats, trends, and LDA topic modeling.
    
    Loads master CSV, computes app-level statistics, monthly trends, and performs
    Latent Dirichlet Allocation (LDA) topic modeling. Can run LDA separately for each
    app and mismatch label, or pooled across all apps per label.
    
    Parameters
    ----------
    master_csv_path : str
        Path to the master CSV file with all processed reviews.
    verbose : bool, optional
        If True, prints all results (app-level stats, trends, topics) to console.
        If False, calculates results silently and returns them. Default is True.
    lda_per_app : bool, optional
        If True, runs LDA separately for each (app, label) combination (6 runs: 3 apps × 2 labels).
        If False, runs LDA once per label pooled across all apps (2 runs total).
        Default is True.
    
    Returns
    -------
    tuple
        If successful, returns (summary_df, trend_df, lda_results).
        
        If lda_per_app=True:
        - lda_results: dict with structure {app_name: {label: topics_list}}
        
        If lda_per_app=False:
        - lda_results: dict with structure {label: topics_list}
    
    Notes
    -----
    - Per-app LDA (lda_per_app=True) reveals app-specific themes for each mismatch type
    - Pooled LDA (lda_per_app=False) shows common patterns across all apps
    - Each LDA run uses its own fitted CountVectorizer and LDA model
    - Results are always saved to RESULT_DIR:
      - app_level_summary.csv: per-app statistics
      - monthly_gap_trend.csv: monthly aggregated gaps
    - verbose flag controls console output only; file output always occurs
    """
    df = load_master_csv(master_csv_path)
    if df is None:
        raise Exception("Incorrectly formatted master csv")

    summary_df = app_level_summary(df)
    summary_df.to_csv(os.path.join(RESULT_DIR, "app_level_summary.csv"), index=False)

    trend_df = monthly_gap_trend(df)
    trend_df.to_csv(os.path.join(RESULT_DIR, "monthly_gap_trend.csv"), index=False)

    # Run LDA per app per label or per label
    if lda_per_app:
        lda_results = {}
        for app_name in DATASET_PATH_DICT.keys():
            lda_results[app_name] = {}
            app_df = df[df["app_name"] == app_name]
            
            for label in ["Inflated", "Deflated"]:
                _, topics = run_lda_on_subset(app_df, label, n_topics=LDA_TOPIC_NUM, n_top_words=LDA_TOPWORD_NUM)
                lda_results[app_name][label] = topics
    else:
        lda_results = {}
        for label in ["Inflated", "Deflated"]:
            _, topics = run_lda_on_subset(df, label, n_topics=LDA_TOPIC_NUM, n_top_words=LDA_TOPWORD_NUM)
            lda_results[label] = topics

    if verbose:
        print("=== App-Level Summary ===")
        print(summary_df)
        
        print("\n=== Monthly Gap Trends ===")
        print(trend_df)
        
        print("\n=== Topic Modeling Results ===")
        if lda_per_app:
            print("(Per App, Per Label)\n")
            for app_name in DATASET_PATH_DICT.keys():
                print(f"--- {app_name} ---")
                for label in ["Inflated", "Deflated"]:
                    print(f"  {label} Topics:")
                    topics = lda_results[app_name][label]
                    if topics:
                        for topic_info in topics:
                            print(f"    Topic {topic_info['topic']}: {', '.join(topic_info['top_words'])}")
                    else:
                        print(f"    [No topics generated - may indicate insufficient data]")
                print()
        else:
            print("(Pooled Across All Apps)\n")
            for label in ["Inflated", "Deflated"]:
                print(f"{label} Topics:")
                topics = lda_results[label]
                if topics:
                    for topic_info in topics:
                        print(f"  Topic {topic_info['topic']}: {', '.join(topic_info['top_words'])}")
                else:
                    print(f"  [No topics generated - may indicate insufficient data]")
                print()

    return summary_df, trend_df, lda_results

### Visualization Functions

#### Scatter Plot: normalized rating vs. sentiment

In [ ]:
def plot_rating_vs_sentiment(df: pd.DataFrame, save_path: str = os.path.join(RESULT_DIR, "scatter_rating_vs_sentiment.png")) -> Axes:
    """Scatter plot of normalized star rating vs. VADER sentiment score, per app.

    Parameters
    ----------
    df : pd.DataFrame
        Scored master dataframe (must contain 'app_name', 'rating', 'sentiment_score').
    save_path : str, optional
        Where to save the figure. Pass None to skip saving.
        Default is RESULT_DIR/scatter_rating_vs_sentiment.png.

    Returns
    -------
    matplotlib.axes.Axes
    """
    fig, ax = plt.subplots(figsize=(8, 6))
    for app_name, group in df.groupby("app_name"):
        normalized_rating = (group["rating"] - 3) / 2
        ax.scatter(normalized_rating, group["sentiment_score"], alpha=0.3, s=15, label=app_name)
    ax.plot([-1, 1], [-1, 1], linestyle="--", color="gray", linewidth=1, label="Perfect agreement")
    ax.set_xlabel("Normalized Star Rating")
    ax.set_ylabel("VADER Sentiment Score")
    ax.set_title("Rating vs. Sentiment by App")
    ax.legend()
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    return ax

#### Stacked bar: Mismatch composition per app

In [ ]:
def plot_mismatch_composition(summary_df: pd.DataFrame, save_path: str = os.path.join(RESULT_DIR, "stacked_bar_mismatch.png")) -> Axes:
    """Stacked bar chart of Inflated/Deflated/Consistent review percentages per app.

    Parameters
    ----------
    summary_df : pd.DataFrame
        Output of app_level_summary(); must contain 'app_name', 'pct_inflated',
        'pct_deflated', 'pct_consistent'.
    save_path : str, optional
        Where to save the figure. Pass None to skip saving.
        Default is RESULT_DIR/stacked_bar_mismatch.png.

    Returns
    -------
    matplotlib.axes.Axes
    """
    stacked_data = summary_df.set_index("app_name")[["pct_inflated", "pct_deflated", "pct_consistent"]]
    fig, ax = plt.subplots(figsize=(8, 6))
    stacked_data.plot(kind="bar", stacked=True, ax=ax, color=["#d9534f", "#5bc0de", "#5cb85c"])
    ax.set_ylabel("% of Reviews")
    ax.set_title("Rating-Sentiment Mismatch Distribution by App")
    ax.legend(title="Mismatch Type")
    plt.xticks(rotation=0)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    return ax

#### Monthly heatmap: Mean gap score, apps x months

In [ ]:
def plot_monthly_gap_heatmap(trend_df: pd.DataFrame, save_path: str = os.path.join(RESULT_DIR, "monthly_gap_heatmap.png")) -> Axes:
    """Heatmap of mean gap score by app (rows) and month (columns).

    Parameters
    ----------
    trend_df : pd.DataFrame
        Output of monthly_gap_trend(); must contain 'app_name', 'year_month', 'mean_gap_score'.
    save_path : str, optional
        Where to save the figure. Pass None to skip saving.
        Default is RESULT_DIR/monthly_gap_heatmap.png.

    Returns
    -------
    matplotlib.axes.Axes
    """
    heatmap_data = trend_df.pivot(index="app_name", columns="year_month", values="mean_gap_score")
    fig, ax = plt.subplots(figsize=(14, 4))
    sns.heatmap(heatmap_data, cmap="RdBu_r", center=0, cbar_kws={"label": "Mean Gap Score"}, ax=ax)
    ax.set_title("Monthly Mean Gap Score by App")
    ax.set_xlabel("Month")
    ax.set_ylabel("App")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    return ax

#### Word clouds: One per (app, mismatch label)

In [ ]:
def plot_lda_wordclouds(lda_results: dict, save_path: str = os.path.join(RESULT_DIR, "wordclouds.png")) -> np.ndarray:
    """Grid of word clouds, one per (app, mismatch label), from LDA topic weights.

    Parameters
    ----------
    lda_results : dict
        Per-app LDA results with structure {app_name: {label: topics_list}},
        as returned by run_full_analysis(lda_per_app=True).
    save_path : str, optional
        Where to save the figure. Pass None to skip saving.
        Default is RESULT_DIR/wordclouds.png.

    Returns
    -------
    numpy.ndarray of matplotlib.axes.Axes
    """
    apps = list(lda_results.keys())
    fig, axes = plt.subplots(len(apps), 2, figsize=(12, 4 * len(apps)))
    if len(apps) == 1:
        axes = axes.reshape(1, -1)

    for row_idx, app_name in enumerate(apps):
        for col_idx, label in enumerate(["Inflated", "Deflated"]):
            topics = lda_results[app_name].get(label)
            ax = axes[row_idx, col_idx]
            if not topics:
                ax.text(0.5, 0.5, "No topics generated", ha="center", va="center")
                ax.axis("off")
                continue

            word_freq = {}
            for topic_info in topics:
                for word, weight in zip(topic_info["top_words"], topic_info["top_weights"]):
                    word_freq[word] = word_freq.get(word, 0) + weight

            wc = WordCloud(width=500, height=350, background_color="white").generate_from_frequencies(word_freq)
            ax.imshow(wc, interpolation="bilinear")
            ax.set_title(f"{app_name} — {label}")
            ax.axis("off")

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    return axes

#### Line Chart: Mean gap score vs months -> for each app

In [ ]:
def plot_monthly_gap_trend(trend_df: pd.DataFrame, save_path: str = os.path.join(RESULT_DIR, "monthly_gap_line_chart.png")) -> Axes:
    """Line chart of monthly mean gap score, one line per app.

    Parameters
    ----------
    trend_df : pd.DataFrame
        Output of monthly_gap_trend(); must contain 'app_name', 'year_month', 'mean_gap_score'.
    save_path : str, optional
        Where to save the figure. Pass None to skip saving.
        Default is RESULT_DIR/monthly_gap_line_chart.png.

    Returns
    -------
    matplotlib.axes.Axes
    """
    fig, ax = plt.subplots(figsize=(14, 5))
    for app_name, group in trend_df.groupby("app_name"):
        group_sorted = group.sort_values("year_month")
        x_values = group_sorted["year_month"].dt.to_timestamp()
        ax.plot(x_values, group_sorted["mean_gap_score"], marker="o", label=app_name)

    ax.axhline(0, color="gray", linestyle="--", linewidth=1)
    ax.set_xlabel("Month")
    ax.set_ylabel("Mean Gap Score")
    ax.set_title("Monthly Mean Gap Score Trend by App")
    ax.legend(title="App")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    return ax

### Main

#### Creating Master CSV

In [ ]:
ONLY_ENGLISH = True  # set True to keep only English-language reviews and skip translation

# Step 1: build (or reuse) the unscored master csv of cleaned reviews
master_df = None
if MASTER_CSV_PATH.exists():
    master_df = load_master_csv(MASTER_CSV_PATH, required_columns=REQUIRED_MASTER_CSV_COLUMNS)
else:
    master_df = build_master_csv(only_english=ONLY_ENGLISH)

if master_df is None:
    raise ValueError("master_df is None")

# Step 2: score the master csv (sentiment + gap scores)
scored_master_df = None
if SCORED_MASTER_CSV_PATH.exists():
    scored_master_df = load_master_csv(SCORED_MASTER_CSV_PATH)
else:
    scored_master_df = score_master_csv(MASTER_CSV_PATH)

if scored_master_df is None:
    raise ValueError("scored_master_df is None")

In [ ]:
del master_df

#### Analysis

In [ ]:
from mfs_sentiment.setup_nltk import ensure_nltk_data

ensure_nltk_data()

In [ ]:
# Run full analysis: app-level stats, monthly trend, LDA topics per app
result = run_full_analysis(
    SCORED_MASTER_CSV_PATH, verbose=True, lda_per_app=True
)

if result is not None:
    summary_df, trend_df, lda_results = result
    del result

#### Visualization

In [ ]:
if scored_master_df is not None:
    plot_rating_vs_sentiment(scored_master_df)

In [ ]:
plot_mismatch_composition(summary_df)

In [ ]:
plot_monthly_gap_heatmap(trend_df)

In [ ]:
plot_lda_wordclouds(lda_results)

In [ ]:
plot_monthly_gap_trend(trend_df)